In [ ]:
# NCrystal notebook settings
# title: Exporting NCrystal materials to ENDF and use them in OpenMC
# menutitle: ENDF files for OpenMC
# shortkey: nd25ncmat2endf
# section: nd2025
# requires: plot, endf, utils, openmc, njoy
# test-parameters: n_particles = 200000; n_particles_broomstick = 100000
# slow: yes

## Introduction [intro]
In this notebook we show how to export ENDF files from NCrystal, process them into ACE files using NJOY, and run simulations in OpenMC. These simulations compare the use of the exported ENDF files in OpenMC with the on-the-fly simulation of thermal scattering in OpenMC using NCrystal.

Download nuclear data library needed by OpenMC (this is not related to NCrystal):

In [ ]:
from ncrystal_notebook_utilities.openmc import download_and_prepare_nndc_data
openmc_xsfile = download_and_prepare_nndc_data()

In [ ]:
import numpy as np
import openmc
openmc.config['cross_sections'] = str(openmc_xsfile)

In [ ]:
#Number of neutrons in the OpenMC simulations below (reduce them for faster, but
#noisier, results):
n_particles = 5000000
n_particles_broomstick = 1000000

## Example: export ENDF-6 file from NCMAT [export]

In [ ]:
import NCrystal as NC

Ncmat2endf can convert most of the files in the NCrystal data library. Some of them cannot be converted either because they include physics that is not implemented in the ENDF-6 format, or require features that have not been yet implemented in ncmat2endf (e.g.: converting scattering kernels for gasses and liquids). The available data library can be listed with `NCrystal.browseFiles()` function, and it is available in the [NCrystal documentation](https://github.com/mctools/ncrystal/wiki/Data-library):

In [ ]:
NC.browseFiles(dump=True, factory='stdlib')

To use it, we need to import the `ncmat2endf` module of NCrystal and call the `ncmat2endf()` function. This function takes as required input a [configuration string](https://github.com/mctools/ncrystal/wiki/CfgRefDoc), which includes the ncmat file to convert as well as parameters that modify the material (temperature, density, which components of the material we want to output). As an example, this converts `Ge3Bi4O12_sg220_BismuthGermanate.ncmat`

In [ ]:
from NCrystal import ncmat2endf
res = ncmat2endf.ncmat2endf('Ge3Bi4O12_sg220_BismuthGermanate.ncmat;temp=293.6K',
                            material_name='BGO', force=True)

The result of the execution is a dictionary with information about the material, which is useful to use these libraries in a radiation transport simulation:

In [ ]:
import pprint
pprint.pprint(res)

In [ ]:
def dump_endf_files(res, nlines=100):
  for entry in res['files']:
      print(entry['file'])
      with open(entry['file']) as f:
          lines = [next(f) for _ in range(nlines)]
      print("".join(lines))
      print('--- file continues ---'.center(80))
      print(80*'*')
dump_endf_files(res)

The file includes generic metadata ("MyLab", "MyLib"). To customize this the `ncmat2endf` module includes the `EndfMetaData` class:

In [ ]:
m = ncmat2endf.EndfMetaData()
help(m)

As an example, we will set the laboratory name to IAEA, the author to Marie Curie, and assign material number 3001 to Bi in BGO, 3002 to Ge in BGO, and 3003 to O in BGO. We will also set the evaluation, revision and distribution dates to the current month:

In [ ]:
m.update_from_dict({'alab':'IAEA', 'auth': 'Marie Curie',
                    'matnum':{'Bi':3001, 'Ge':3002, 'O':3003},
                    'libname':'IAEA-TSL'})
m.set_all_dates_as_now()
res = ncmat2endf.ncmat2endf('Ge3Bi4O12_sg220_BismuthGermanate.ncmat;temp=293.6K',
                            endf_metadata=m,
                            material_name='BGO', force=True)
dump_endf_files(res, nlines=10)

The same thing can be achieved using the command line interface:

In [ ]:
!ncrystal_ncmat2endf 'Ge3Bi4O12_sg220_BismuthGermanate.ncmat;temp=293.6K'\
      --name 'BGO' -f --now -m alab:IAEA -m auth:"Marie Curie"\
      -m matnum:"Bi:3001,Ge:3002,O:3003" -m libname:IAEA-TSL\
      --index result.json

In [ ]:
!cat result.json

## Processing the ENDF files and adding them to OpenMC [processing]

In [ ]:
def convert_openmc_ncrystal_to_ace(cfg, matname, elastic_mode):
  from ncrystal_notebook_utilities.endf2ace import convert_endf_tsl_to_ace

  m = openmc.Material()

  # Generate ENDF-6 file
  from NCrystal import ncmat2endf
  import os

  outxml = openmc.config['cross_sections']
  openmc_datalib = openmc.data.DataLibrary.from_xml(outxml)
  from pathlib import Path
  reporoot = Path(outxml).parent
  h5outdir = reporoot
  res = ncmat2endf.ncmat2endf(cfg, material_name=matname,
                              force=True, elastic_mode=elastic_mode)
  m.temperature = float(res['temperature'])
  m.set_density('g/cc', res['density'])

  for entry in res['files']:
    m.add_element(entry['component'], entry['fraction'], 'ao')
    endf_fn = entry['file']
    name = endf_fn.replace('.endf', '').replace('tsl_', 'c_')
    ace_fn = name+f'-{m.temperature:.1f}K.ace'
    print(f'Converting {endf_fn} to {ace_fn}')
    # Process ENDF-6 file into ACE file
    ace_fn, xsdir_fn = convert_endf_tsl_to_ace(endf_fn, ace_filename=ace_fn)
    f_ace = Path(ace_fn)
    # Convert to HDF5 and register data file
    print(f'Adding {name} to the library...')
    f_h5 = h5outdir.joinpath( '%s.h5'%f_ace.stem )
    try:
        os.remove(f_h5)
    except OSError:
        pass
    assert not f_h5.is_file() and f_h5.parent.is_dir()
    openmc.data.ThermalScattering.from_ace(f_ace, name=name).export_to_hdf5(f_h5)
    print(f"Generated {f_h5}")
    openmc_datalib.register_file(f_h5)
    m.add_s_alpha_beta(name)
  openmc_datalib.export_to_xml(outxml)
  return m

In [ ]:
cfg = 'AlN_sg186_AluminumNitride.ncmat;temp=293.6K'
m1 = openmc.Material.from_ncrystal(cfg)
m1.name = 'AlN (NCrystal)'
print(m1)
m2 = convert_openmc_ncrystal_to_ace(cfg, 'AlN', elastic_mode='mixed')
m2.name = 'AlN (ncmat2endf)'
print(m2)

fig = openmc.plot_xs({m1:['total'], m2:['total']})
fig.axes[0].set_title("AlN comparison")

## Use OpenMC to compare on-the-fly and processed libraries [onthefly]


In this example we will calculate the angular distribution from polypropylene in a broomstick problem.

In [ ]:
def create_broomstick_model(broomstick_mat, E0=0.95, Emin=0, Emax=20e6,
                            NParticles=n_particles_broomstick, broomstick_diameter=0.001,
                            broomstick_length=10000):
    p1 = openmc.ZPlane(z0=-broomstick_length/2)
    p2 = openmc.ZPlane(z0=+broomstick_length/2)
    c1 = openmc.ZCylinder(r=broomstick_diameter)
    s1 = openmc.Sphere(r=broomstick_length*1.5, boundary_type='vacuum')
    cell1 = openmc.Cell(region=+p1&-p2&-c1, fill=broomstick_mat)
    cell2 = openmc.Cell(region=~(+p1&-p2&-c1)&-s1, fill=None)
    u1 = openmc.Universe(cells=[cell1, cell2])
    geometry = openmc.Geometry(root=u1)

    source = openmc.IndependentSource()
    source.space  = openmc.stats.Point(xyz=[0,0,-broomstick_length/2])
    source.angle  = openmc.stats.Monodirectional(reference_uvw=[0,0,1])
    source.energy = openmc.stats.Discrete([E0], [1.0])

    settings = openmc.Settings()
    settings.source = source
    settings.run_mode = 'fixed source'
    settings.batches = 10
    settings.particles = NParticles
    settings.temperature = {'method':'nearest', 'tolerance':1000}

    tally1 = openmc.Tally()
    filter1 = openmc.SurfaceFilter([c1])
    filter2 = openmc.EnergyFilter([Emin, Emax])
    filter3 = openmc.PolarFilter(np.arccos(np.linspace(1,-1,101)))
    tally1.scores = ['current']
    tally1.name = 'angdist'
    tally1.filters = [filter1, filter2, filter3]
    tallies = openmc.Tallies([tally1])

    return openmc.model.Model(geometry=geometry, settings=settings, tallies=tallies)

In [ ]:
cfg = 'Polypropylene_C3H6.ncmat'
material_name = 'Polypropylene'
spfile1, spfile2 = 'results_ncrystal.h5', 'results_ncmat2endf.h5'

m1 = openmc.Material.from_ncrystal(cfg)
print(m1)
E0 = 0.1  # eV
model = create_broomstick_model(m1, E0=0.95, Emin=0.27, Emax=0.28) # , E0=0.1, Emin=0.05, Emax=0.06)
spfile = model.run(output=True)
import os
os.rename(spfile, spfile1)

m2 = convert_openmc_ncrystal_to_ace(cfg, material_name, elastic_mode='mixed')
print(m2)
model = create_broomstick_model(m2, E0=0.95, Emin=0.27, Emax=0.28) # , E0=0.1, Emin=0.05, Emax=0.06)
spfile = model.run(output=True)
os.rename(spfile, spfile2)

In [ ]:
for spfile in (spfile1, spfile2):
  with openmc.StatePoint(spfile) as sp:
    tal = sp.get_tally(name='angdist')
  tal.remove_filter(tal.find_filter(openmc.EnergyFilter))
  df = tal.get_pandas_dataframe()
  bin_high = df['polar high [rad]'].values
  bin_low = df['polar low [rad]'].values
  angdist_values = df['mean'].values/(bin_high - bin_low)
  import matplotlib.pyplot as plt
  plt.step(np.rad2deg(bin_high), angdist_values, label=spfile)
plt.legend()
plt.xlabel('Polar angle [deg]')
plt.ylabel('density')

## Compare the "scaled" and "mixed" approximations for vanadium [vanadium]

In [ ]:
def pencil_beam_model(m1, E0, N, Emin=0, Emax=20e6):
    """Return an openmc.Model() object for a monoenergetic pencil
     beam hitting a 1 mm sphere filled with the material m1
     and compute angular distribution for the
     Debye-Scherrer cones"""

    # Material definition
    materials = openmc.Materials([m1])

    # Geometry definition

    sample_sphere = openmc.Sphere(r=0.1)
    outer_sphere = openmc.Sphere(r=100, boundary_type="vacuum")
    cell1 = openmc.Cell(region=-sample_sphere, fill=m1)
    cell2_region = +sample_sphere & -outer_sphere
    cell2 = openmc.Cell(region=cell2_region, fill=None)
    geometry = openmc.Geometry([cell1, cell2])

    # Source definition

    source = openmc.IndependentSource()
    source.space = openmc.stats.Point((0, 0, -20))
    source.angle = openmc.stats.Monodirectional(reference_uvw=(0, 0, 1))
    source.energy = openmc.stats.Discrete([E0], [1.0])

    # Execution settings

    settings = openmc.Settings()
    settings.source = source
    settings.run_mode = "fixed source"
    settings.batches = 10
    settings.particles = N

    # Tally definition

    tally1 = openmc.Tally(name="angular distribution")
    tally1.scores = ["current"]
    filter1 = openmc.SurfaceFilter(sample_sphere)
    filter2 = openmc.PolarFilter(np.linspace(0, np.pi, 180+1))
    filter3 = openmc.CellFromFilter(cell1)
    filter4 = openmc.EnergyFilter([Emin, Emax])
    tally1.filters = [filter1, filter2, filter3, filter4]
    tallies = openmc.Tallies([tally1])

    return openmc.Model(geometry, materials, settings, tallies)

def plot_tallies(spfile1, spfile2, title=None):
    import matplotlib.pyplot as plt
    plt.figure()
    results = {}
    for spfile in [spfile1, spfile2]:
        with openmc.StatePoint(spfile) as sp:
            tal = sp.get_tally(name='angular distribution')
            tal.remove_filter(tal.find_filter(openmc.EnergyFilter))
            df = tal.get_pandas_dataframe()
        bin_high = df['polar high [rad]'].values
        bin_low = df['polar low [rad]'].values
        angdist_values = df['mean'].values/(bin_high - bin_low)

        plt.step(np.rad2deg(bin_high), angdist_values, label=spfile)
        results[spfile] = angdist_values
    plt.yscale('log')
    plt.ylabel('density')
    plt.title(title)
    plt.xlabel('Polar angle [deg]')
    plt.legend()
    plt.show()
    plt.figure()
    plt.step(np.rad2deg(bin_high), results[spfile2]/results[spfile1])
    plt.plot([np.rad2deg(bin_high)[0], np.rad2deg(bin_high)[-1]], [0.5, 0.5], 'k--')
    plt.plot([np.rad2deg(bin_high)[0], np.rad2deg(bin_high)[-1]], [1.5, 1.5], 'k--')
    plt.ylabel('ratio')
    plt.ylim(0, 2)
    plt.title(title)
    plt.xlabel('Polar angle [deg]')
    plt.show()

In [ ]:
def compare_ncrystal_ncmat2endf(cfg, material_name, elastic_mode='scaled', E0 = 0.01, title='XS comparison'):
  m1 = openmc.Material.from_ncrystal(cfg)
  m1.name = material_name + '(NCrystal)'
  model = pencil_beam_model(m1, E0, n_particles)
  spfile = model.run(output=False)
  import os
  spfile1, spfile2 = 'results_ncrystal.h5', 'results_ncmat2endf.h5'
  os.rename(spfile, spfile1)

  m2 = convert_openmc_ncrystal_to_ace(cfg, material_name, elastic_mode=elastic_mode)
  m2.name = material_name + f'(ncmat2endf - {elastic_mode})'
  model = pencil_beam_model(m2, E0, n_particles)
  spfile = model.run(output=False)
  os.rename(spfile, spfile2)
  plot_tallies(spfile1, spfile2, title=material_name)
  
  fig = openmc.plot_xs({m1:['total'], m2:['total']})
  fig.axes[0].set_title(title)

In [ ]:
cfg = 'V_sg229.ncmat;temp=293.6K'
material_name = 'Vanadium'
elastic_mode='scaled'
E0 = 0.0045

compare_ncrystal_ncmat2endf(cfg, material_name=material_name, 
                            E0=E0, elastic_mode=elastic_mode,
                            title='Vanadium comparison (scaled approximation)' )

In [ ]:
cfg = 'V_sg229.ncmat;temp=293.6K'
material_name = 'Vanadium'
elastic_mode='mixed'

compare_ncrystal_ncmat2endf(cfg, material_name=material_name, 
                            E0=E0, elastic_mode=elastic_mode,
                            title='Vanadium comparison (mixed approximation)' )

In [ ]:
import NCrystal.minimc as ncmmc
result = ncmmc.run(cfg, scenario=f'{E0}eV pencil on 1cm sphere',
                   enginecfg='tally=theta;tallybins=theta:1800:0:180')

result.tally('theta').plot(rebin_factor=10,logy=True)

## Compare a simulation for Bismuth Germanate [cmpsim]

In [ ]:
cfg = 'Ge3Bi4O12_sg220_BismuthGermanate.ncmat;temp=293.6K'
material_name = 'BGO'
elastic_mode='mixed'
compare_ncrystal_ncmat2endf(cfg, material_name, elastic_mode=elastic_mode, title='Bismuth Germanate')